# Recherche de règle : BM25 contre embeddings (E5, BGE-M3) contre hybride

**Avant de lancer** : Exécution > Modifier le type d'exécution > **GPU T4**.

Ce notebook mesure Recall@1, Recall@3, Recall@5 et MRR sur les 1 912 clauses de test.
Référence à battre (BM25, mesurée sur votre PC) : **Recall@5 = 0,684**.

Préparation sur votre PC, dans le dossier du projet :
```
python scripts/make_colab_zip.py
```
Cela crée `colab_retrieval.zip`, à envoyer à l'étape 2.

## 1. Installation

In [1]:
!pip -q install sentence-transformers rank-bm25 pyyaml

## 2. Envoyer `colab_retrieval.zip`

In [2]:
from google.colab import files
up = files.upload()          # choisir colab_retrieval.zip
!unzip -oq colab_retrieval.zip
!ls src/retrieval data/rules data/processed

Saving colab_retrieval.zip to colab_retrieval.zip
data/processed:
clauses_test.jsonl

data/rules:
rules_contractuelles.json

src/retrieval:
build_index.py	eval_retrieval.py  __init__.py	 rules.py
corpus.py	index.py	   law_watch.py


In [3]:
import torch
print("GPU disponible :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun (activer le GPU)")

GPU disponible : True | Tesla T4


## 3. E5 multilingue (BM25, dense et hybride sont mesurés d'un coup)

In [4]:
!python -m src.retrieval.eval_retrieval --dense --model intfloat/multilingual-e5-base --no-mlflow --out retrieval_e5_base.json

1912 clauses de test, 41 règles
modules.json: 100% 387/387 [00:00<00:00, 1.72MB/s]
README.md: 100% 179k/179k [00:00<00:00, 173MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 222kB/s]
config.json: 100% 694/694 [00:00<00:00, 3.48MB/s]

model.safetensors: downloading bytes:  31% 350M/1.11G [00:02<00:03, 201MB/s, 29.7MB/s  ]
model.safetensors: downloading bytes:  40% 447M/1.11G [00:02<00:03, 218MB/s, 37.8MB/s  ]
model.safetensors: downloading bytes:  51% 564M/1.11G [00:03<00:03, 142MB/s, 46.4MB/s  ]
model.safetensors: downloading bytes:  53% 590M/1.11G [00:03<00:03, 137MB/s, 47.1MB/s  ]
model.safetensors: downloading bytes:  56% 624M/1.11G [00:04<00:03, 146MB/s, 48.9MB/s  ]
model.safetensors: downloading bytes:  58% 641M/1.11G [00:04<00:03, 148MB/s, 50.2MB/s  ]
model.safetensors: downloading bytes:  61% 677M/1.11G [00:04<00:02, 157MB/s, 52.2MB/s  ]
model.safetensors: downloading bytes:  66% 739M/1.11G [00:04<00:02, 158MB/s, 54.7MB/s  ]
model.safetensors: reconstructing file: 

## 4. BGE-M3 (autre encodeur, pour comparer)

In [5]:
!python -m src.retrieval.eval_retrieval --dense --model BAAI/bge-m3 --no-mlflow --out retrieval_bge_m3.json

1912 clauses de test, 41 règles
modules.json: 100% 349/349 [00:00<00:00, 1.66MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 588kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 36.8MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 263kB/s]
config.json: 100% 687/687 [00:00<00:00, 4.45MB/s]

pytorch_model.bin: downloading bytes:   9% 196M/2.27G [00:01<00:10, 193MB/s, 16.4MB/s  ]
pytorch_model.bin: downloading bytes:  10% 232M/2.27G [00:02<00:13, 151MB/s, 20.3MB/s  ]
pytorch_model.bin: downloading bytes:  20% 444M/2.27G [00:02<00:07, 259MB/s, 37.3MB/s  ]
pytorch_model.bin: downloading bytes:  41% 926M/2.27G [00:04<00:04, 331MB/s, 72.3MB/s  ]
pytorch_model.bin: downloading bytes:  45% 1.02G/2.27G [00:07<00:20, 61.7MB/s, 62.3MB/s  ]
pytorch_model.bin: downloading bytes:  66% 1.50G/2.27G [00:13<00:02, 268MB/s, 74.7MB/s  ]
pytorch_model.bin: reconstructing file:  68% 1.54G/2.27G [00:15<00:08, 90.4MB/s, 95.1MB/s  ]
pytorch_model.bin: downloading bytes: 100% 1.52G

## 5. Tableau récapitulatif

In [6]:
import json, glob
from pathlib import Path
import pandas as pd

rows = []
for f in sorted(glob.glob("reports/retrieval_*.json")):
    d = json.load(open(f, encoding="utf-8"))
    for mode, r in d["resultats"].items():
        rows.append({"encodeur": (d["encodeur"] or "-").split("/")[-1], "méthode": mode,
                     **{k: round(v, 3) for k, v in r.items() if k != "n"}})
df = pd.DataFrame(rows)
df = df[~((df["méthode"] == "bm25") & df.duplicated(subset=["méthode", "recall@5", "mrr"]))]   # BM25 apparaît une seule fois
df.sort_values("recall@5", ascending=False).reset_index(drop=True)

,encodeur,méthode,recall@1,recall@3,recall@5,mrr
0,bge-m3,dense,0.463,0.700,0.787,0.584
1,bge-m3,hybrid,0.458,0.703,0.785,0.586
2,multilingual-e5-base,hybrid,0.411,0.658,0.765,0.547
3,multilingual-e5-base,dense,0.379,0.634,0.730,0.513
4,bge-m3,bm25,0.358,0.595,0.684,0.481


## 6. Récupérer les résultats

Les deux fichiers `.json` sont à placer dans `reports/` de votre projet, puis, sur votre PC :
```
python -m src.retrieval.eval_retrieval --log-only reports/retrieval_e5_base.json
python -m src.retrieval.eval_retrieval --log-only reports/retrieval_bge_m3.json
```
Ils apparaissent alors dans MLflow (expérience `02-recherche-regles`), à côté de BM25.

In [7]:
files.download("reports/retrieval_e5_base.json")
files.download("reports/retrieval_bge_m3.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>